# **Data Cleaning**

In [160]:
# import necessary libraries
import pandas as pd
import re

In [161]:
# function to clean the seasons column
def expand_seasons(seasons_str):
    if pd.isna(seasons_str) or seasons_str.strip() == "":
        return []

    clean_str = re.sub(r'\[.*?\]', '', seasons_str)
    clean_str = clean_str.replace("present", "2026")
    
    years = set()
    parts = [p.strip() for p in clean_str.split(',')]

    for part in parts:
        if '–' in part:
            try:
                start, end = [int(y.strip()) for y in part.split('–')]
                years.update(range(start, end + 1))
            except:
                continue
        else:
            if part.isdigit():
                years.add(int(part))

    years.discard(2026)

    return sorted(years)

In [162]:
# function to check if 2025 is in the seasons list
def has_2025(seasons):
    if isinstance(seasons, list):
        return 2025 in seasons
    return False

## **Drivers**
_API_

In [163]:
df_drivers = pd.read_csv("../data_acquisition/drivers_api.csv")

# check for null values
df_drivers.isnull().values.any()

np.True_

In [164]:
# delete drivers with null values
df_drivers = df_drivers.dropna(subset=["number", "dateOfBirth"])

In [165]:
# explicit type casting
df_drivers['driverId'] = df_drivers['driverId'].astype('string')
df_drivers['name'] = df_drivers['name'].astype('string')
df_drivers['surname'] = df_drivers['surname'].astype('string')
df_drivers['number'] = pd.to_numeric(df_drivers['number'], errors='coerce').astype('Int64')
df_drivers['dateOfBirth'] = pd.to_datetime(df_drivers['dateOfBirth'], errors='coerce')

# check type and null values
df_drivers.info()

<class 'pandas.DataFrame'>
Index: 21 entries, 0 to 29
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   driverId     21 non-null     string        
 1   number       21 non-null     Int64         
 2   name         21 non-null     string        
 3   surname      21 non-null     string        
 4   dateOfBirth  21 non-null     datetime64[us]
dtypes: Int64(1), datetime64[us](1), string(3)
memory usage: 1.4 KB


In [166]:
df_drivers.to_csv("drivers_api_clean.csv", index=False)

_Scraping_

In [167]:
df_drivers = pd.read_csv("../data_acquisition/drivers_wiki.csv")

# expand seasons and keep only drivers active in 2025
df_drivers['seasons'] = df_drivers['seasons'].apply(expand_seasons)
df_drivers = df_drivers[df_drivers['seasons'].apply(has_2025)]

# clean driverName 
df_drivers['driverName'] = df_drivers['driverName'].str.replace(r'[|*~]', '', regex=True).str.strip()

# Pulizia colonne numeriche
championships_clean = []
for val in df_drivers['championships']:
    match = re.search(r'\d+', str(val))  # cerca il primo numero
    if match:
        championships_clean.append(int(match.group()))
    else:
        championships_clean.append(0)
df_drivers['championships'] = championships_clean

print(df_drivers.shape)  # dimensioni del DataFrame


(21, 11)


In [168]:
# explicit type casting
df_drivers['driverName'] = df_drivers['driverName'].astype('string')
df_drivers['nationality'] = df_drivers['nationality'].astype('string')
df_drivers['championships'] = pd.to_numeric(df_drivers['championships'], errors='coerce').astype('Int64')
df_drivers['entries']        = pd.to_numeric(df_drivers['entries'], errors='coerce').astype('Int64')
df_drivers['starts']         = pd.to_numeric(df_drivers['starts'], errors='coerce').astype('Int64')
df_drivers['poles']          = pd.to_numeric(df_drivers['poles'], errors='coerce').astype('Int64')
df_drivers['wins']           = pd.to_numeric(df_drivers['wins'], errors='coerce').astype('Int64')
df_drivers['podiums']        = pd.to_numeric(df_drivers['podiums'], errors='coerce').astype('Int64')
df_drivers['fastestLaps']    = pd.to_numeric(df_drivers['fastestLaps'], errors='coerce').astype('Int64')
df_drivers['points'] = pd.to_numeric(df_drivers['points'], errors='coerce')

# check type and null values
df_drivers.info()

<class 'pandas.DataFrame'>
Index: 21 entries, 11 to 817
Data columns (total 11 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   driverName     21 non-null     string 
 1   nationality    21 non-null     string 
 2   seasons        21 non-null     object 
 3   championships  21 non-null     Int64  
 4   entries        21 non-null     Int64  
 5   starts         21 non-null     Int64  
 6   poles          21 non-null     Int64  
 7   wins           21 non-null     Int64  
 8   podiums        21 non-null     Int64  
 9   fastestLaps    21 non-null     Int64  
 10  points         21 non-null     float64
dtypes: Int64(7), float64(1), object(1), string(2)
memory usage: 2.6+ KB


In [169]:
df_drivers.to_csv("drivers_wiki_clean.csv", index=False)

## **Constructors**
_Scraping_

In [170]:
df_constructors = pd.read_csv("../data_acquisition/constructors_wiki.csv")

# expand seasons and keep only constructors active in 2025
df_constructors["seasons"] = df_constructors["seasons"].apply(expand_seasons)
df_constructors = df_constructors[df_constructors['seasons'].apply(has_2025)]

# remove columns that appear in only in one of the two original tables
df_constructors.drop(columns=["engine", "basedIn", "antecedentTeams"], inplace=True)

# clean constructor and licensedIn columns
df_constructors["constructor"] = df_constructors["constructor"].str.replace(r"\[.*?\]", "", regex=True)
df_constructors["licensedIn"] = df_constructors["licensedIn"].str.replace(r"\[.*?\]", "", regex=True)

# keep only the main name of the constructor
df_constructors["constructor"] = df_constructors["constructor"].str.split("/").str[0]

# separate multiple countries in licensedIn
df_constructors["licensedIn"] = df_constructors["licensedIn"].str.replace(r"(?<=[a-z])(?=[A-Z])", "; ", regex=True)

In [171]:
# explicit type casting
df_constructors['constructor'] = df_constructors['constructor'].astype('string') 
df_constructors['licensedIn'] = df_constructors['licensedIn'].astype('string') 
df_constructors['racesEntered'] = pd.to_numeric(df_constructors['racesEntered'], errors='coerce').astype('Int64')  # type Int64
df_constructors['racesStarted'] = pd.to_numeric(df_constructors['racesStarted'], errors='coerce').astype('Int64')  # type Int64
df_constructors['drivers'] = pd.to_numeric(df_constructors['drivers'], errors='coerce').astype('Int64')  # type Int64
df_constructors['totalEntries'] = pd.to_numeric(df_constructors['totalEntries'], errors='coerce').astype('Int64')  # type Int64
df_constructors['wins'] = pd.to_numeric(df_constructors['wins'], errors='coerce').astype('Int64')  # type Int64
df_constructors['points'] = pd.to_numeric(df_constructors['points'], errors='coerce').astype('float')  # type float
df_constructors['poles'] = pd.to_numeric(df_constructors['poles'], errors='coerce').astype('Int64')  # type Int64
df_constructors['fastestLaps'] = pd.to_numeric(df_constructors['fastestLaps'], errors='coerce').astype('Int64')  # type float
df_constructors['podiums'] = pd.to_numeric(df_constructors['podiums'], errors='coerce').astype('Int64')  # type Int64
df_constructors['wcc'] = pd.to_numeric(df_constructors['wcc'], errors='coerce').astype('Int64')  # type Int64
df_constructors['wdc'] = pd.to_numeric(df_constructors['wdc'], errors='coerce').astype('Int64')  # type Int64

# check type and null values
df_constructors.info()

<class 'pandas.DataFrame'>
Index: 10 entries, 0 to 116
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   constructor   10 non-null     string 
 1   licensedIn    10 non-null     string 
 2   seasons       10 non-null     object 
 3   racesEntered  10 non-null     Int64  
 4   racesStarted  10 non-null     Int64  
 5   drivers       10 non-null     Int64  
 6   totalEntries  10 non-null     Int64  
 7   wins          10 non-null     Int64  
 8   points        10 non-null     float64
 9   poles         10 non-null     Int64  
 10  fastestLaps   10 non-null     Int64  
 11  podiums       10 non-null     Int64  
 12  wcc           10 non-null     Int64  
 13  wdc           10 non-null     Int64  
dtypes: Int64(10), float64(1), object(1), string(2)
memory usage: 1.5+ KB


In [172]:
df_constructors.to_csv("constructor_wiki_clean.csv", index=False)

# **Races**
_API_

In [173]:
df_races = pd.read_csv("../data_acquisition/races_api.csv")

# delete 'Z' from time column
df_races['time'] = df_races['time'].str.replace('Z', '', regex=False)
df_races['sprintTime'] = df_races['sprintTime'].str.replace('Z', '', regex=False)

In [174]:
# explicit type casting
df_races['round'] = pd.to_numeric(df_races['round'], errors='coerce').astype('Int64')
df_races['raceName'] = df_races['raceName'].astype('string')
df_races['raceDateTime'] = pd.to_datetime(df_races['date'].astype(str) + ' ' + df_races['time'].astype(str), errors='coerce')
df_races['sprintDateTime'] = pd.to_datetime(df_races['sprintDate'].astype(str) + ' ' + df_races['sprintTime'].astype(str), errors='coerce')
df_races['circuitId'] = df_races['circuitId'].astype('string')

df_races.drop(columns=["time", "date", "sprintTime", "sprintDate"], inplace=True)

# check type and null values
df_races.info() #there are null values, but it's expected since not every round included a sprint

<class 'pandas.DataFrame'>
RangeIndex: 24 entries, 0 to 23
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   round           24 non-null     Int64         
 1   raceName        24 non-null     string        
 2   circuitId       24 non-null     string        
 3   raceDateTime    24 non-null     datetime64[us]
 4   sprintDateTime  6 non-null      datetime64[us]
dtypes: Int64(1), datetime64[us](2), string(2)
memory usage: 1.7 KB


In [175]:
df_races.to_csv("races.csv", index=False)

# **Circuits**
_Scraping_

In [176]:
df_circuits = pd.read_csv("../data_acquisition/circuits_wiki.csv")

#expand seasons and keep only circuits active in 2025
df_circuits['seasons'] = df_circuits['seasons'].apply(expand_seasons)
df_circuits = df_circuits[df_circuits['seasons'].apply(has_2025)]

# clean circuitName
df_circuits['circuitName'] = df_circuits['circuitName'].str.replace(r'[|*~]', '', regex=True).str.strip()

# drop map column
df_circuits.drop(columns=["map"], inplace=True)

# clean direction column
def clean_direction(direction_str):
    if pd.isna(direction_str):
        return ''
    direction_str = str(direction_str).strip()
    if direction_str == 'Clockwise':
        return 'Clockwise'
    elif direction_str == 'Anti-clockwise':
        return 'Anti-clockwise'
    else:
        return 'Both'

df_circuits['direction'] = df_circuits['direction'].apply(clean_direction)

# clean length_km column in order to keep only the km value
df_circuits['length_km'] = df_circuits['length_km'].str.extract(r'([\d\.]+)\s*km')[0]

In [177]:
# explicit type casting
df_circuits['circuitName'] = df_circuits['circuitName'].astype('string')
df_circuits['type'] = df_circuits['type'].astype('string')
df_circuits['direction'] = df_circuits['direction'].astype('string')
df_circuits['city'] = df_circuits['city'].astype('string')
df_circuits['country'] = df_circuits['country'].astype('string')
df_circuits['length_km'] = pd.to_numeric(df_circuits['length_km'], errors='coerce').astype('float')
df_circuits['turns'] = pd.to_numeric(df_circuits['turns'], errors='coerce').astype('Int64')
df_circuits['grandPrix'] = df_circuits['grandPrix'].astype('string')
df_circuits['grandPrixHeld'] = pd.to_numeric(df_circuits['grandPrixHeld'], errors='coerce').astype('Int64')

# check type and null values
df_circuits.info()

<class 'pandas.DataFrame'>
Index: 24 entries, 3 to 76
Data columns (total 10 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   circuitName    24 non-null     string 
 1   type           24 non-null     string 
 2   direction      24 non-null     string 
 3   city           24 non-null     string 
 4   country        24 non-null     string 
 5   length_km      24 non-null     float64
 6   turns          24 non-null     Int64  
 7   grandPrix      24 non-null     string 
 8   seasons        24 non-null     object 
 9   grandPrixHeld  24 non-null     Int64  
dtypes: Int64(2), float64(1), object(1), string(6)
memory usage: 4.3+ KB


In [178]:
df_circuits.to_csv("circuits_wiki_clean.csv", index=False)

# **Results**
_API_

In [179]:
def str_to_timedelta(t):
    if pd.isna(t) or t.strip() == "":
        return pd.NA
    t = t.strip()
    if t.startswith("+"):
        t = t[1:]
    parts = t.split(":")
    try:
        if len(parts) == 3:  # HH:MM:SS.sss
            h, m, s = parts
            return pd.to_timedelta(f"{h}:{m}:{s}")
        elif len(parts) == 2:  # MM:SS.sss
            m, s = parts
            return pd.to_timedelta(f"0:{m}:{s}")
        else:  # solo secondi
            return pd.to_timedelta(float(parts[0]), unit='s')
    except:
        return pd.NA

def absolutize_times(df):
    df = df.copy()
    for r in df['round'].unique():
        mask = df['round'] == r
        round_times = df.loc[mask, 'time']
        if round_times.isna().all():
            continue
        first_time = round_times.iloc[0]
        # creiamo un indice booleano per tutti tranne il primo
        idx = mask & (df.index != round_times.index[0])
        # sommiamo solo ai valori non null
        df.loc[idx, 'time'] = df.loc[idx, 'time'] + first_time
    return df

In [180]:
df_results = pd.read_csv("../data_acquisition/results_api.csv")

df_results['round'] = pd.to_numeric(df_results['round'], errors='coerce').astype('Int64')
df_results['driverId'] = df_results['driverId'].astype('string')
df_results['constructorId'] = df_results['constructorId'].astype('string')
df_results['grid'] = pd.to_numeric(df_results['grid'], errors='coerce').astype('Int64')
df_results['position'] = pd.to_numeric(df_results['position'], errors='coerce').astype('Int64')
df_results['points'] = pd.to_numeric(df_results['points'], errors='coerce').astype('Int64')
df_results['status'] = df_results['status'].astype('string')
df_results['laps'] = pd.to_numeric(df_results['laps'], errors='coerce').astype('Int64')

df_results['time'] = df_results['time'].apply(str_to_timedelta)
df_results['fastestLap'] = df_results['fastestLap'].apply(str_to_timedelta)

df_results = absolutize_times(df_results)

# check type and null values
df_results.info() # there are null values, but it's expected since not every driver finishes every race

<class 'pandas.DataFrame'>
RangeIndex: 479 entries, 0 to 478
Data columns (total 10 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   round          479 non-null    Int64 
 1   driverId       479 non-null    string
 2   constructorId  479 non-null    string
 3   grid           479 non-null    Int64 
 4   position       479 non-null    Int64 
 5   points         479 non-null    Int64 
 6   status         479 non-null    string
 7   laps           479 non-null    Int64 
 8   time           424 non-null    object
 9   fastestLap     457 non-null    object
dtypes: Int64(5), object(2), string(3)
memory usage: 50.0+ KB


In [181]:
df_results.to_csv("results_api_clean.csv", index=False)

# **Sprints**
_API_

In [182]:
df_sprints = pd.read_csv("../data_acquisition/sprints_api.csv")

df_sprints['round'] = pd.to_numeric(df_sprints['round'], errors='coerce').astype('Int64')
df_sprints['driverId'] = df_sprints['driverId'].astype('string')
df_sprints['constructorId'] = df_sprints['constructorId'].astype('string')
df_sprints['grid'] = pd.to_numeric(df_sprints['grid'], errors='coerce').astype('Int64')
df_sprints['position'] = pd.to_numeric(df_sprints['position'], errors='coerce').astype('Int64')
df_sprints['points'] = pd.to_numeric(df_sprints['points'], errors='coerce').astype('Int64')
df_sprints['status'] = df_sprints['status'].astype('string')
df_sprints['laps'] = pd.to_numeric(df_sprints['laps'], errors='coerce').astype('Int64')

df_sprints['time'] = df_sprints['time'].apply(str_to_timedelta)
df_sprints['fastestLap'] = df_sprints['fastestLap'].apply(str_to_timedelta)

df_sprints = absolutize_times(df_sprints)

# check type and null values
df_sprints.info() # there are null values, but it's expected since not every driver finishes every race

<class 'pandas.DataFrame'>
RangeIndex: 120 entries, 0 to 119
Data columns (total 10 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   round          120 non-null    Int64 
 1   driverId       120 non-null    string
 2   constructorId  120 non-null    string
 3   grid           120 non-null    Int64 
 4   position       120 non-null    Int64 
 5   points         120 non-null    Int64 
 6   status         120 non-null    string
 7   laps           120 non-null    Int64 
 8   time           109 non-null    object
 9   fastestLap     116 non-null    object
dtypes: Int64(5), object(2), string(3)
memory usage: 12.6+ KB


In [183]:
df_sprints.to_csv("sprints_api_clean.csv", index=False)